# 01 · Ingest & prepare

Ingest sample drone frame/track metadata into a Delta table and prepare a
context-ready DataFrame. Uses the bundled `notebooks/sample_data/` JSON if you
have no data yet.

In [ ]:
# ---- CONFIG (the only cell you must edit) --------------------------------
dbutils.widgets.text("input_delta_path", "drone.tracks", "Input Delta table/path")
dbutils.widgets.text("output_delta_path", "drone.inference_results", "Output Delta table/path")
dbutils.widgets.text("model_name", "visdrone-yolov8x", "DVSA model name")
dbutils.widgets.text("batch_size", "64", "Batch size")
dbutils.widgets.text("checkpoint_location", "/Volumes/drone/_chk", "Checkpoint (streaming)")
dbutils.widgets.text("secret_scope", "dvsa", "Databricks Secret scope")

INPUT_DELTA = dbutils.widgets.get("input_delta_path")
OUTPUT_DELTA = dbutils.widgets.get("output_delta_path")
MODEL_NAME = dbutils.widgets.get("model_name")
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))
CHECKPOINT = dbutils.widgets.get("checkpoint_location")
SECRET_SCOPE = dbutils.widgets.get("secret_scope")

In [ ]:
from dvsa_databricks_connector import ingest_videos_to_delta, prepare_context_from_delta

# Point this at your landing zone; the repo ships small samples you can upload
# to a Volume/DBFS for a first run.
SOURCE_PATH = "/Volumes/drone/landing/tracks/*.json"

ingest_videos_to_delta(spark, SOURCE_PATH, INPUT_DELTA, fmt="json", mode="overwrite")
df = prepare_context_from_delta(spark, INPUT_DELTA)
display(df)

In [ ]:
# Sanity-check the Delta -> DVSA context mapping on the first row (pure, offline).
from dvsa_databricks_connector import row_to_context
print(row_to_context(df.limit(1).collect()[0].asDict(recursive=True)))